# 07. Construcción de documentos de resumen diario

Este notebook construye la estructura documental que se utilizará posteriormente en MongoDB para almacenar un resumen por cada día del conjunto de datos.

Los datos minuto a minuto se recuperan desde PostgreSQL y se transforman en un documento diario que incluye información sobre cobertura temporal, irradiancia, meteorología, calidad de las mediciones, imputaciones y campos reservados para futuros resultados de modelos.

En esta fase no se insertan documentos en MongoDB. El objetivo es validar la estructura y los cálculos utilizando inicialmente una única fecha.

In [30]:
import os
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [31]:
load_dotenv()

postgres_host = os.getenv("POSTGRES_HOST")
postgres_port = os.getenv("POSTGRES_PORT")
postgres_database = os.getenv("POSTGRES_DB")
postgres_user = os.getenv("POSTGRES_USER")
postgres_password = os.getenv("POSTGRES_PASSWORD")

required_variables = {
    "POSTGRES_HOST": postgres_host,
    "POSTGRES_PORT": postgres_port,
    "POSTGRES_DB": postgres_database,
    "POSTGRES_USER": postgres_user,
    "POSTGRES_PASSWORD": postgres_password,
}

missing_variables = [
    variable
    for variable, value in required_variables.items()
    if not value
]

if missing_variables:
    raise ValueError(
        "Faltan las siguientes variables en el archivo .env: "
        + ", ".join(missing_variables)
    )

print("Variables de PostgreSQL cargadas correctamente.")

Variables de PostgreSQL cargadas correctamente.


In [32]:
postgres_uri = (
    f"postgresql+psycopg2://{postgres_user}:{postgres_password}"
    f"@{postgres_host}:{postgres_port}/{postgres_database}"
)

engine = create_engine(postgres_uri)

In [33]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1;"))
    print(f"Conexión correcta con PostgreSQL: {result.scalar() == 1}")

Conexión correcta con PostgreSQL: True


In [34]:
test_date = "2023-07-15"

print(f"Fecha seleccionada: {test_date}")

Fecha seleccionada: 2023-07-15


In [35]:
def load_daily_measurements(
    engine,
    date: str,
    schema: str = "solar",
    table: str = "measurements",
) -> pd.DataFrame:
    """
    Recupera desde PostgreSQL todas las mediciones correspondientes
    a una fecha concreta.

    Parameters
    ----------
    engine: Motor de conexión SQLAlchemy.
    date: Fecha en formato YYYY-MM-DD.
    schema: Esquema de PostgreSQL.
    table: Tabla que contiene las mediciones.

    Returns
    -------
    pd.DataFrame: Mediciones correspondientes al día solicitado.
    """

    query = text(
        f"""
        SELECT *
        FROM {schema}.{table}
        WHERE fecha >= :start_date
          AND fecha < CAST(:start_date AS DATE) + INTERVAL '1 day'
        ORDER BY fecha;
        """
    )

    df_day = pd.read_sql_query(
        query,
        con=engine,
        params={"start_date": date},
    )

    if df_day.empty:
        raise ValueError(
            f"No se han encontrado registros para la fecha {date}."
        )

    return df_day

In [36]:
df_day = load_daily_measurements(
    engine=engine,
    date=test_date,
)

print(f"Número de registros: {len(df_day):,}")
print(f"Fecha mínima: {df_day['fecha'].min()}")
print(f"Fecha máxima: {df_day['fecha'].max()}")
print(f"Número de columnas: {df_day.shape[1]}")
print(f"Fechas duplicadas: {df_day['fecha'].duplicated().sum()}")

Número de registros: 1,440
Fecha mínima: 2023-07-15 00:00:00
Fecha máxima: 2023-07-15 23:59:00
Número de columnas: 29
Fechas duplicadas: 0


In [37]:
def to_python_number(value):
    """
    Convierte valores NumPy o pandas a tipos nativos de Python.
    """

    if pd.isna(value):
        return None

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    return value

In [38]:
def numeric_summary(series: pd.Series) -> dict:
    """
    Calcula las principales estadísticas descriptivas de una serie.
    """

    valid_values = series.dropna()

    if valid_values.empty:
        return {
            "media": None,
            "mediana": None,
            "minimo": None,
            "maximo": None,
            "desviacion_estandar": None,
            "nulos": int(series.isna().sum()),
        }

    return {
        "media": to_python_number(valid_values.mean()),
        "mediana": to_python_number(valid_values.median()),
        "minimo": to_python_number(valid_values.min()),
        "maximo": to_python_number(valid_values.max()),
        "desviacion_estandar": to_python_number(valid_values.std()),
        "nulos": int(series.isna().sum()),
    }

In [39]:
def categorical_distribution(series: pd.Series) -> dict:
    """
    Calcula la distribución absoluta y porcentual de una variable categórica.
    """

    counts = series.value_counts(dropna=False).sort_index()
    total = len(series)

    absolute = {}
    percentage = {}

    for category, count in counts.items():
        category_key = "null" if pd.isna(category) else str(category)

        absolute[category_key] = int(count)
        percentage[category_key] = round(
            float(count / total * 100),
            4,
        )

    return {
        "frecuencia": absolute,
        "porcentaje": percentage,
    }

In [40]:
def binary_summary(series: pd.Series) -> dict:
    """
    Resume una variable indicadora binaria.

    El valor 0 representa la ausencia de la condición y el valor 1
    representa su presencia.
    """

    numeric_series = pd.to_numeric(
        series,
        errors="coerce",
    )

    invalid_values = numeric_series[
        numeric_series.notna()
        & ~numeric_series.isin([0, 1])
    ]

    if not invalid_values.empty:
        raise ValueError(
            "La serie contiene valores distintos de 0 y 1."
        )

    total = len(numeric_series)
    positives = int(numeric_series.eq(1).sum())
    negatives = int(numeric_series.eq(0).sum())
    nulls = int(numeric_series.isna().sum())

    valid_total = positives + negatives

    percentage = (
        round(positives / valid_total * 100, 4)
        if valid_total > 0
        else None
    )

    return {
        "valor_0": negatives,
        "valor_1": positives,
        "nulos": nulls,
        "porcentaje_valor_1": percentage,
    }

In [41]:
def build_processing_summary(df: pd.DataFrame) -> dict:
    """
    Resume los indicadores de imputación meteorológica y ausencia
    original de irradiancias.
    """

    return {
        "imputacion_meteorologica": {
            "descripcion": (
                "Indica si al menos una variable meteorológica "
                "del registro ha sido imputada."
            ),
            **binary_summary(df["var_meteo_imp"]),
        },
        "irradiancia_original_nula": {
            "descripcion": (
                "Indica si el registro presentaba al menos una "
                "irradiancia nula antes del tratamiento."
            ),
            **binary_summary(df["irr_null"]),
        },
    }

In [42]:
def build_quality_summary(df: pd.DataFrame) -> dict:
    """
    Construye la distribución diaria de los códigos de calidad
    de GHI, DNI y DHI.
    """

    return {
        "codigo_ghi": categorical_distribution(
            df["codigo_ghi"]
        ),
        "codigo_dni": categorical_distribution(
            df["codigo_dni"]
        ),
        "codigo_dhi": categorical_distribution(
            df["codigo_dhi"]
        ),
    }

In [43]:
def build_daily_document(
    df: pd.DataFrame,
    dataset_version: str = "v3",
) -> dict:
    """
    Construye un documento de resumen diario compatible con MongoDB.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame correspondiente exclusivamente a un día.
    dataset_version : str
        Versión del dataset utilizada.

    Returns
    -------
    dict
        Documento diario preparado para su posterior inserción
        en MongoDB.
    """

    if df.empty:
        raise ValueError(
            "No se puede construir un documento a partir "
            "de un DataFrame vacío."
        )

    df = df.copy()

    # Las fechas almacenadas proceden originalmente de una serie UTC.
    df["fecha"] = pd.to_datetime(
        df["fecha"],
        errors="coerce",
    )

    if df["fecha"].isna().any():
        raise ValueError(
            "Existen valores de fecha que no se han podido convertir."
        )

    unique_dates = df["fecha"].dt.date.unique()

    if len(unique_dates) != 1:
        raise ValueError(
            "El DataFrame contiene registros correspondientes "
            "a más de un día."
        )

    document_date = pd.Timestamp(unique_dates[0])

    registros_dia = int(
        df["periodo_solar"].eq("dia").sum()
    )

    registros_noche = int(
        df["periodo_solar"].eq("noche").sum()
    )

    now_utc = datetime.now(timezone.utc)

    daily_document = {
        "fecha": datetime(
            year=document_date.year,
            month=document_date.month,
            day=document_date.day,
            tzinfo=timezone.utc,
        ),

        "dataset": {
            "version": dataset_version,
            "origen": "PostgreSQL",
            "tabla_origen": "solar.measurements",
        },

        "periodo": {
            "ano": int(document_date.year),
            "mes": int(document_date.month),
            "dia": int(document_date.day),
            "dia_semana": int(document_date.dayofweek),
        },

        "cobertura": {
            "numero_registros": int(len(df)),
            "fecha_inicio": df["fecha"].min().to_pydatetime(),
            "fecha_fin": df["fecha"].max().to_pydatetime(),
            "fechas_duplicadas": int(
                df["fecha"].duplicated().sum()
            ),
            "periodo_solar": {
                "registros_dia": registros_dia,
                "registros_noche": registros_noche,
            },
        },

        "irradiancia": {
            "ghi": numeric_summary(df["ghi"]),
            "dni": numeric_summary(df["dni"]),
            "dhi": numeric_summary(df["dhi"]),
            "ghi_estimado": numeric_summary(
                df["ghi_estimado"]
            ),
        },

        "meteorologia": {
            "temperatura": numeric_summary(
                df["temperatura"]
            ),
            "humedad_relativa": numeric_summary(
                df["humedad_relativa"]
            ),
            "velocidad_viento": numeric_summary(
                df["velocidad_viento"]
            ),
            "direccion_viento": numeric_summary(
                np.degrees(np.arctan2(
                    df["direccion_viento_sin"], df["direccion_viento_cos"]
                    )) % 360
            )
        },

        "variables_fisicas": {
            "elevacion_solar": numeric_summary(
                df["elevacion_solar"]
            ),
            "error_balance": numeric_summary(
                df["error_balance"]
            ),
            "error_balance_abs": numeric_summary(
                df["error_balance_abs"]
            ),
            "error_balance_rel": numeric_summary(
                df["error_balance_rel"]
            ),
        },

        "calidad": build_quality_summary(df),

        "procesamiento": build_processing_summary(df),

        "graficas": {
            "curvas_solares": {
                "disponible": False,
                "ruta": None,
                "formato": None,
                "fecha_generacion": None,
            },
            "calidad_meteorologia": {
                "disponible": False,
                "ruta": None,
                "formato": None,
                "fecha_generacion": None,
            },
        },

        "resultados_modelos": {
            "disponible": False,
            "ejecuciones": [],
        },

        "anomalias_modelo": {
            "disponible": False,
            "eventos": [],
        },

        "explicacion_automatica": {
            "disponible": False,
            "texto": None,
            "modelo_generador": None,
            "fecha_generacion": None,
        },

        "metadatos": {
            "schema_version": "1.0",
            "created_at": now_utc,
            "updated_at": now_utc,
        },
    }

    return daily_document

In [44]:
daily_document = build_daily_document(
    df=df_day,
    dataset_version="v3",
)

print("Documento diario construido correctamente.")

Documento diario construido correctamente.


In [45]:
from pprint import pprint

pprint(
    daily_document,
    sort_dicts=False,
    width=120,
)

{'fecha': datetime.datetime(2023, 7, 15, 0, 0, tzinfo=datetime.timezone.utc),
 'dataset': {'version': 'v3', 'origen': 'PostgreSQL', 'tabla_origen': 'solar.measurements'},
 'periodo': {'ano': 2023, 'mes': 7, 'dia': 15, 'dia_semana': 5},
 'cobertura': {'numero_registros': 1440,
               'fecha_inicio': datetime.datetime(2023, 7, 15, 0, 0),
               'fecha_fin': datetime.datetime(2023, 7, 15, 23, 59),
               'fechas_duplicadas': 0,
               'periodo_solar': {'registros_dia': 867, 'registros_noche': 573}},
 'irradiancia': {'ghi': {'media': 6.622047222222222,
                         'mediana': 0.0,
                         'minimo': 0.0,
                         'maximo': 236.333,
                         'desviacion_estandar': 30.830683952388345,
                         'nulos': 0},
                 'dni': {'media': 4.247625694444444,
                         'mediana': 0.0,
                         'minimo': 0.0,
                         'maximo': 355.75,
     

In [46]:
assert (
    daily_document["cobertura"]["numero_registros"]
    == len(df_day)
)

periodo_summary = daily_document[
    "cobertura"
]["periodo_solar"]

assert (
    periodo_summary["registros_dia"]
    + periodo_summary["registros_noche"]
    == len(df_day)
)

assert (
    daily_document["cobertura"]["fechas_duplicadas"]
    == df_day["fecha"].duplicated().sum()
)

print("Cobertura diaria validada correctamente.")

Cobertura diaria validada correctamente.


In [47]:
for target in [
    "codigo_ghi",
    "codigo_dni",
    "codigo_dhi",
]:
    target_frequency = daily_document[
        "calidad"
    ][target]["frecuencia"]

    assert sum(target_frequency.values()) == len(df_day), (
        f"La distribución de {target} no suma "
        "el total de registros diarios."
    )

print("Distribuciones de calidad validadas correctamente.")

Distribuciones de calidad validadas correctamente.


In [48]:
processing = daily_document["procesamiento"]

for indicator_name in [
    "imputacion_meteorologica",
    "irradiancia_original_nula",
]:
    indicator = processing[indicator_name]

    indicator_total = (
        indicator["valor_0"]
        + indicator["valor_1"]
        + indicator["nulos"]
    )

    assert indicator_total == len(df_day), (
        f"El indicador {indicator_name} no suma "
        "el total de registros."
    )

print("Indicadores binarios validados correctamente.")

Indicadores binarios validados correctamente.


In [49]:
from bson import BSON

try:
    BSON.encode(daily_document)
    print("El documento es compatible con BSON.")
except Exception as exc:
    print("El documento contiene tipos incompatibles con MongoDB.")
    raise exc

El documento es compatible con BSON.


## Resultado

Se ha recuperado desde PostgreSQL un día completo de mediciones y se ha transformado en un documento de resumen diario compatible con MongoDB.

El documento incluye la cobertura temporal del día, estadísticas descriptivas de irradiancia y meteorología, variables físicas, distribuciones de los códigos de calidad e indicadores binarios sobre imputación meteorológica y presencia original de irradiancias nulas.

También se han preparado estructuras vacías para incorporar posteriormente las rutas de las gráficas, los resultados diarios de los modelos, las anomalías detectadas y las explicaciones automáticas. En esta fase todavía no se ha insertado ningún documento en MongoDB.